# Volatility index curves (e.g. VIX term structure)

Deep-dive reference: **forward levels of a volatility index** across horizon — not equity option implied vol at a single strike.

## Concept

Indices like **VIX** quote **implied volatility** of a model portfolio of options; the **term structure** is how that index level is expected to evolve by horizon. A **`PriceCurve`** built with **`kind="vol_index"`** stores **`(t, level)`** knots where the level is in **vol points** (e.g. 18 for an 18% vol index reading); levels must be non-negative.

## API walkthrough

Use **`price(t)`** to read the curve at year fraction `t` (or pass a date). `kind` reports `"vol_index"` and `spot_price` the front level.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import PriceCurve
base = date(2024, 1, 1)
vix = PriceCurve(
    "VIX",
    base,
    [
        (0.0, 16.0),
        (1 / 12, 17.5),
        (0.25, 19.0),
        (0.5, 20.0),
        (1.0, 22.0),
    ],
    kind="vol_index",
    day_count="act_365f",
)
print("curve:", vix)
print("kind:", vix.kind, "spot level:", vix.spot_price)
print("Term structure (implied index forward level, stylized vol points):")
for t in (0.0, 1 / 12, 0.25, 0.5, 1.0):
    print(f"  t={t:.4f}y -> level = {vix.price(t):.4f}")


## Practical example

Compare **front** vs **1Y** forward index level — often used in **vol-of-vol** or **dispersion** discussions at a high level.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import PriceCurve
base = date(2024, 1, 1)
vix = PriceCurve("VIX", base, [(0.0, 18.0), (1.0, 21.0)], kind="vol_index", day_count="act_365f")
front = vix.price(0.0)
one_y = vix.price(1.0)
print(f"Front level: {front:.2f}")
print(f"1Y level:    {one_y:.2f}")
print(f"Spread (1Y - spot curve front): {one_y - front:.2f} vol points")


## Takeaways

- This is a **1D term structure of index levels**, not a **strike / expiry vol surface** for one underlying.
- **`price(t)`** interpolates the **stylized** market view of future index values; `MarketContext.get_vol_index_curve` returns the same `PriceCurve`.
- For **options on a single name**, you typically need a **`VolSurface`** or instrument-specific vol inputs.